In [9]:
import numpy as np

from implementations import (
    mean_squared_error_gd,
    mean_squared_error_sgd,
    least_squares,
    ridge_regression,
    logistic_regression,
    reg_logistic_regression,
)

from helpers import load_csv_data, create_csv_submission

DATA_PATH = "data/dataset/"

SEED = 67
VALIDATION_FRACTION = 0.20

# settings
MSE_GD_MAX_ITERS = 500
MSE_GD_GAMMA = 0.03

MSE_SGD_MAX_ITERS = 100_000
MSE_SGD_GAMMA = 0.0001

RIDGE_LAMBDA = 1e-5

LOGISTIC_MAX_ITERS = 10000
LOGISTIC_GAMMA = 0.03

REG_LOGISTIC_MAX_ITERS = 2000
REG_LOGISTIC_GAMMA = 0.1
REG_LOGISTIC_LAMBDA = 1e-5

In [10]:
x_train_raw, x_test_raw, y_train, train_ids, test_ids = load_csv_data(
    DATA_PATH,
    sub_sample=False
)

y_train = np.asarray(y_train).ravel()

print("Train:", x_train_raw.shape)
print("Test:", x_test_raw.shape)
print("Labels:", np.unique(y_train, return_counts=True))

NEG_TO_POS_RATIO = 3   # negatives kept per positive in the fit set (1 = 50/50)

# Stratified split
rng = np.random.default_rng(SEED)

positive_idx = np.where(y_train == 1)[0]
negative_idx = np.where(y_train == -1)[0]

rng.shuffle(positive_idx)
rng.shuffle(negative_idx)

n_pos_val = int(len(positive_idx) * VALIDATION_FRACTION)
n_neg_val = int(len(negative_idx) * VALIDATION_FRACTION)

# Validation: natural distribution, untouched
val_idx = np.concatenate([
    positive_idx[:n_pos_val],
    negative_idx[:n_neg_val]
])

# Fit: all positives + random subset of negatives
pos_fit_idx = positive_idx[n_pos_val:]
neg_fit_idx = negative_idx[n_neg_val:]

n_neg_keep = min(len(neg_fit_idx), NEG_TO_POS_RATIO * len(pos_fit_idx))
neg_fit_idx = rng.choice(neg_fit_idx, size=n_neg_keep, replace=False)

train_idx = np.concatenate([pos_fit_idx, neg_fit_idx])

rng.shuffle(train_idx)
rng.shuffle(val_idx)

x_fit_raw = x_train_raw[train_idx]
y_fit = y_train[train_idx]

x_val_raw = x_train_raw[val_idx]
y_val = y_train[val_idx]

print("Fit split:", x_fit_raw.shape, "| positives:", (y_fit == 1).mean().round(3))
print("Validation split:", x_val_raw.shape, "| positives:", (y_val == 1).mean().round(3))

Train: (328135, 321)
Test: (109379, 321)
Labels: (array([-1,  1]), array([299160,  28975]))
Fit split: (92720, 321) | positives: 0.25
Validation split: (65627, 321) | positives: 0.088


In [11]:
def fit_preprocessor(X):

    # Remove features that are entirely missing
    keep = np.any(np.isfinite(X), axis=0)
    X = X[:, keep].astype(float, copy=True)

    # Treat non-finite values as missing
    X[~np.isfinite(X)] = np.nan

    means = np.nanmean(X, axis=0)
    stds = np.nanstd(X, axis=0)

    # Avoid division by zero or NaN in case of constant features
    stds[~np.isfinite(stds) | (stds == 0)] = 1.0

    return keep, means, stds


def transform_features(X, keep, means, stds):
    X = X[:, keep].astype(float, copy=True)
    X[~np.isfinite(X)] = np.nan

    # Impute missing values using training means
    X = np.where(np.isnan(X), means, X)

    X = (X - means) / stds

    # Add bias term
    X = np.column_stack([np.ones(X.shape[0]), X])

    return X


# Validation preprocessing: fit on the fitting split only
keep_val, means_val, stds_val = fit_preprocessor(x_fit_raw)

X_fit = transform_features(
    x_fit_raw, keep_val, means_val, stds_val
)

X_val = transform_features(
    x_val_raw, keep_val, means_val, stds_val
)

print("Processed fit:", X_fit.shape)
print("Processed validation:", X_val.shape)

Processed fit: (92720, 322)
Processed validation: (65627, 322)


In [12]:
def fit_mse_gd(y, X):
    return mean_squared_error_gd(
        y, X, np.zeros(X.shape[1]),
        max_iters=MSE_GD_MAX_ITERS,
        gamma=MSE_GD_GAMMA
    )


def fit_mse_sgd(y, X):
    return mean_squared_error_sgd(
        y, X, np.zeros(X.shape[1]),
        max_iters=MSE_SGD_MAX_ITERS,
        gamma=MSE_SGD_GAMMA
    )


def fit_least_squares(y, X):
    return least_squares(y, X)


def fit_ridge(y, X):
    return ridge_regression(y, X, RIDGE_LAMBDA)


def fit_logistic(y, X):
    return logistic_regression(
        y, X, np.zeros(X.shape[1]),
        max_iters=LOGISTIC_MAX_ITERS,
        gamma=LOGISTIC_GAMMA
    )


def fit_reg_logistic(y, X):
    return reg_logistic_regression(
        y, X, REG_LOGISTIC_LAMBDA, np.zeros(X.shape[1]),
        max_iters=REG_LOGISTIC_MAX_ITERS,
        gamma=REG_LOGISTIC_GAMMA
    )


models = {
    "mse_gd": fit_mse_gd,
    "mse_sgd": fit_mse_sgd,
    "least_squares": fit_least_squares,
    "ridge": fit_ridge,
    "logistic": fit_logistic,
    "reg_logistic": fit_reg_logistic,
}

In [13]:
def find_best_accuracy_threshold(y_true, scores):

    candidates = np.unique(
        np.quantile(scores, np.linspace(0, 1, 1001))
    )

    best_threshold = 0.0
    best_accuracy = -1.0

    for threshold in candidates:
        predictions = np.where(scores >= threshold, 1, -1)
        accuracy = np.mean(predictions == y_true)

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_threshold = threshold

    return best_threshold, best_accuracy


def predict_labels(scores, threshold):
    return np.where(scores >= threshold, 1, -1)

In [14]:
# redefine best_f1_treshold since cannot import from notebook
def classification_metrics(y_true, scores, threshold=0.0):
    predictions = np.where(scores >= threshold, 1, -1)

    accuracy = float(np.mean(predictions == y_true))
    tp = int(np.sum((y_true == 1) & (predictions == 1)))
    fp = int(np.sum((y_true == -1) & (predictions == 1)))
    fn = int(np.sum((y_true == 1) & (predictions == -1)))

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "positive_predictions": int(np.sum(predictions == 1)),
    }


def best_f1_threshold(y_true, scores, n_candidates=201):
    # Quantiles focus candidate thresholds where the model's scores actually lie.
    thresholds = np.unique(np.quantile(scores, np.linspace(0, 1, n_candidates)))
    best_threshold = 0.0
    best_metrics = classification_metrics(y_true, scores, best_threshold)

    for threshold in thresholds:
        metrics = classification_metrics(y_true, scores, float(threshold))
        if metrics["f1"] > best_metrics["f1"]:
            best_threshold = float(threshold)
            best_metrics = metrics

    return best_threshold, best_metrics

In [15]:
results = {}

# Fit preprocessing on ALL labelled training data for final models
full_keep, full_means, full_stds = fit_preprocessor(x_train_raw)

X_full = transform_features(
    x_train_raw, full_keep, full_means, full_stds
)

X_test = transform_features(
    x_test_raw, full_keep, full_means, full_stds
)
# Balanced version of the full training set (same ratio as the fit set)
pos_all = np.where(y_train == 1)[0]
neg_all = np.where(y_train == -1)[0]
neg_keep = rng.choice(
    neg_all,
    size=min(len(neg_all), NEG_TO_POS_RATIO * len(pos_all)),
    replace=False
)
full_bal_idx = rng.permutation(np.concatenate([pos_all, neg_keep]))

X_full_bal = X_full[full_bal_idx]
y_full_bal = y_train[full_bal_idx]

for name, fit_model in models.items():
    print("MODEL:", name)

    # 1. Fit on the fitting split
    w_val, val_train_loss = fit_model(y_fit, X_fit)

    # 2. Get validation scores and choose F1 threshold
    val_scores = X_val @ w_val

    if not np.isfinite(val_scores).all():
        print("Validation scores contain NaN or infinity.")

    threshold, val_metrics = best_f1_threshold(y_val, val_scores)

    print(f"Validation training loss: {val_train_loss:.8f}")
    print(f"Chosen threshold:         {threshold:.8f}")
    print(f"Validation F1:            {val_metrics['f1']:.5f}")
    print(f"Validation precision:     {val_metrics['precision']:.5f}")
    print(f"Validation recall:        {val_metrics['recall']:.5f}")
    print(f"Validation accuracy:      {val_metrics['accuracy']:.5f}")

    # 3. Retrain on the entire labelled dataset
    w_final, full_train_loss = fit_model(y_full_bal, X_full_bal)

    # 4. Predict test labels
    predictions = X_test @ w_final

    if not np.isfinite(predictions).all():
        print("Predictions contains NaN or infinity.")

    test_predictions = predict_labels(predictions, threshold)

    # 5. Save one submission per model
    output_path = f"submission_{name}.csv"

    create_csv_submission(
        test_ids,
        test_predictions,
        str(output_path)
    )

    results[name] = {
        "threshold": threshold,
        "validation_f1": val_metrics["f1"],
        "validation_accuracy": val_metrics["accuracy"],
        "validation_loss": val_train_loss,
        "full_training_loss": full_train_loss,
        "positive_predictions": int(np.sum(test_predictions == 1)),
        "negative_predictions": int(np.sum(test_predictions == -1)),
        "file": str(output_path),
    }


    print("Full training loss:", full_train_loss)
    print("Test predictions:", len(test_predictions))
    print("Predicted classes:", np.unique(
        test_predictions, return_counts=True
    ))
    print("Saved:", output_path)


print("\nFinished generating submissions.")

MODEL: mse_gd
Validation training loss: 0.26771132
Chosen threshold:         -0.12959256
Validation F1:            0.41397
Validation precision:     0.32121
Validation recall:        0.58205
Validation accuracy:      0.85448
Full training loss: 0.2675806987272204
Test predictions: 109379
Predicted classes: (array([-1,  1]), array([91967, 17412]))
Saved: submission_mse_gd.csv
MODEL: mse_sgd
Validation training loss: 0.28181051
Chosen threshold:         -0.00695240
Validation F1:            0.40108
Validation precision:     0.31121
Validation recall:        0.56393
Validation accuracy:      0.85128
Full training loss: 0.2722514285064441
Test predictions: 109379
Predicted classes: (array([-1,  1]), array([95699, 13680]))
Saved: submission_mse_sgd.csv
MODEL: least_squares
Validation training loss: 0.26549141
Chosen threshold:         -0.06151763
Validation F1:            0.41907
Validation precision:     0.35185
Validation recall:        0.51803
Validation accuracy:      0.87318
Full train

In [16]:

def get_validation_accuracy(item):
    name, result = item
    return result["validation_accuracy"]


print(
    f"{'Model':<18} {'Val acc':>9} {'Threshold':>12} "
    f"{'Positives':>11}"
)

sorted_results = sorted(
    results.items(),
    key=get_validation_accuracy,
    reverse=True
)

for name, result in sorted_results:
    print(
        f"{name:<18} "
        f"{result['validation_accuracy']:>9.4f} "
        f"{result['threshold']:>12.5f} "
        f"{result['positive_predictions']:>11} "
    )


Model                Val acc    Threshold   Positives
least_squares         0.8732     -0.06152       14124 
ridge                 0.8671     -0.08537       15200 
reg_logistic          0.8624     -0.20475       16258 
logistic              0.8596     -0.23816       16745 
mse_gd                0.8545     -0.12959       17412 
mse_sgd               0.8513     -0.00695       13680 
